# 🚀 Module 5: Modern Python (3.11–3.13+)

**Мета:** Зрозуміти ключові зміни в останніх версіях CPython, що стосуються пам’яті та продуктивності.

## 1. Immortal Objects (PEP 683, Python 3.12+)

Проблема: Об’єкти `None`, `True`, `False`, малі цілі числа — ніколи не видаляються. Але їхній `ob_refcnt` постійно змінюється, що:

1. **Знищує Copy-on-Write (CoW)** після `fork()` (критично для Gunicorn, uWSGI)
2. **Створює cache-line bouncing** в багатопоточному коді

**Рішення PEP 683:** Встановити `ob_refcnt` в спеціальне значення, що ніколи не змінюється.

In [1]:
import sys

print(f"Python version: {sys.version}")
print()

# Перевіримо refcount для «безсмертних» об'єктів
immortals = [None, True, False, 0, 1, 256, -5]

for obj in immortals:
    rc = sys.getrefcount(obj)
    print(f"{str(obj):>6}  refcount = {rc}")

print("\nУ Python 3.12+ refcount для цих об'єктів може бути")
print("дуже великим (>2^30) — це означає, що об'єкт 'immortal'.")

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]

  None  refcount = 3221225560
  True  refcount = 3221225472
 False  refcount = 3221225472
     0  refcount = 3221225472
     1  refcount = 3221225472
   256  refcount = 3221225472
    -5  refcount = 3221225472

У Python 3.12+ refcount для цих об'єктів може бути
дуже великим (>2^30) — це означає, що об'єкт 'immortal'.


In [2]:
# Small Integer Cache: числа від -5 до 256 кешуються (interning)
a = 256
b = 256
print(f"256 is 256? {a is b}")  # True — той самий об'єкт

a = 257
b = 257
print(f"257 is 257? {a is b}")  # False в REPL, True в скрипті (компілятор може оптимізувати)

# Демонстрація межі
print(f"\nid(256) = {id(256)}")
print(f"id(257) = {id(257)}")
print(f"id(-5)  = {id(-5)}")
print(f"id(-6)  = {id(-6)}  (не кешоване)")

256 is 256? True
257 is 257? False

id(256) = 140707663680600
id(257) = 1964653280880
id(-5)  = 140707663672248
id(-6)  = 1964653280752  (не кешоване)


## 2. Adaptive Interpreter (PEP 659, Python 3.11+)

Python 3.11 додав **Specializing Adaptive Interpreter** («Quickening»). Це означає:

1. Байткод **спостерігає** за типами операндів при кожному виконанні
2. Якщо типи стабільні — байткод **спеціалізується** (напр., `BINARY_ADD` → `BINARY_ADD_INT`)
3. Якщо тип змінився — відкат до generic інструкції

**Вплив на пам’ять:** Додатковий inline cache прямо в байткоді (невеликий overhead, значний приріст швидкості).

In [3]:
import dis
import sys

def add_ints(a, b):
    return a + b

def concat_strings(a, b):
    return a + b

# Дивимось на байткод до спеціалізації
print("=== Bytecode for add_ints ===")
dis.dis(add_ints)

# Викликаємо кілька разів, щоб пройшла спеціалізація
for _ in range(100):
    add_ints(1, 2)

print("\n=== Bytecode AFTER specialization (adaptive=True) ===")
# У Python 3.11+ dis може показати спеціалізовані інструкції
try:
    dis.dis(add_ints, adaptive=True)
except TypeError:
    print("(adaptive parameter requires Python 3.11+)")
    dis.dis(add_ints)

=== Bytecode for add_ints ===
  4           RESUME                   0

  5           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE

=== Bytecode AFTER specialization (adaptive=True) ===
  4           RESUME_CHECK             0

  5           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


## 3. Per-Interpreter GIL (PEP 684, Python 3.12+)

**Класична проблема:** GIL (Global Interpreter Lock) дозволяє лише одному потоку виконувати Python байткод одночасно.

**PEP 684:** Кожний суб-інтерпретатор отримує **власний GIL**, що дозволяє справжню паралельність.

**Вплив на пам’ять:**
- Кожний інтерпретатор має власний набір вбудованих об’єктів
- Immortal objects спільні між інтерпретаторами (не копіюються)
- Зменшує потребу в multiprocessing (який дублює всю пам’ять)

In [4]:
import threading
import time
import sys

# Демонстрація GIL: CPU-bound задачі не прискорюються від потоків
def cpu_bound(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

# Однопотоково
start = time.perf_counter()
cpu_bound(N)
cpu_bound(N)
single = time.perf_counter() - start

# Багатопотоково
start = time.perf_counter()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
threaded = time.perf_counter() - start

print(f"Single-threaded: {single:.3f}s")
print(f"Two threads:     {threaded:.3f}s")
print(f"Speedup:         {single/threaded:.2f}x")
print(f"\nЯкщо speedup ≈ 1.0 — це GIL в дії.")
print(f"Python: {sys.version}")

Single-threaded: 0.416s
Two threads:     0.467s
Speedup:         0.89x

Якщо speedup ≈ 1.0 — це GIL в дії.
Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


## 4. Free-Threaded Python (PEP 703, Python 3.13+)

Python 3.13 додав **експериментальний** режим без GIL (`--disable-gil` / `python3.13t`).

**Важливо:** Для free-threaded режиму потрібна **спеціальна збірка** CPython (з суфіксом `t`, наприклад `python3.14t`). Звичайна збірка завжди має GIL увімкненим.

**Як це впливає на пам'ять:**
- `ob_refcnt` стає **атомарним** (atomic operations замість простого `++`/`--`)
- Додано **biased reference counting** для зменшення contention
- Garbage Collector повинен працювати без stop-the-world пауз

In [5]:
import sys

# Перевірка: чи зібрано Python без GIL?
has_gil = True
if hasattr(sys, '_is_gil_enabled'):
    has_gil = sys._is_gil_enabled()
    print(f"GIL enabled: {has_gil}")
else:
    print("sys._is_gil_enabled() not available (Python < 3.13)")
    print(f"GIL is always enabled in Python {sys.version_info.major}.{sys.version_info.minor}")

# Визначення build конфігурації
print(f"\nBuild info: {sys.version}")
print(f"Implementation: {sys.implementation.name}")

GIL enabled: True

Build info: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Implementation: cpython


## 5. Підсумок

| Фіча | PEP | Версія | Вплив на пам’ять |
|------|-----|--------|------------------|
| Immortal Objects | 683 | 3.12+ | Зменшує CoW, прибирає refcount churn |
| Adaptive Interpreter | 659 | 3.11+ | Невеликий оверхед, але прискорює виконання |
| Per-Interpreter GIL | 684 | 3.12+ | Альтернатива multiprocessing (менше RAM) |
| Free-Threaded | 703 | 3.13+ | Atomic refcount, biased RC |